# 13 — External Candidate Coverage Gate

**Purpose.** Measure whether held-out structures are reachable from EXTERNAL structure sources (COCONUT now, PubChem as a membership estimate) under a leakage-free protocol: **validation protocol v2**.

* **Coverage only.** Nothing is trained, tuned or ranked here. No Recall@25 / Recall@100 / MRR from any ranker is a Gate-A number. Ranking of external candidates starts in notebook 14 (analog baseline).
* **Read-only.** It reads the finalized universe `candidates/universe/` (TRAIN ∪ COCONUT, modern or **adopted** legacy), the Stage-A builds `candidates/universe_work/` (modern ACTIVE pointers **and** legacy `_done/<SRC>-chunk-NNNNN.json` markers), the v2 regime manifest of notebook 11b, the panels and the optional PubChem membership index. It never rebuilds the universe.
* **Stops early** with `REGIME_REBUILD_REQUIRED` when the v2 regime manifest is missing or bound to another identity (universe, external sources, evidence, panels, dev queries, split, policy, code).
* **Outputs.** Everything goes to `reports/candidate_coverage/` on Drive (`coverage.report_dir`).

Run order: 10 → 11a (legacy adoption) → 11b (regimes v2) → **this notebook** → follow the decision printed at the end.

## Why the old C2 Recall@All could be trivial (leakage explanation)

Validation molecules come from TRAIN. The finalized universe is TRAIN ∪ COCONUT, so:

1. **TRAIN provenance.** Every held-out truth is in the universe through its TRAIN record. "Truth in universe" is ~100% by construction. It measures mass accuracy, not database coverage.
2. **Selection on the outcome.** The v1 C2 regime was *selected* on having a non-TRAIN source and split 34 / 33 / 33. v2 regimes are eligibility-based with natural counts (notebook 11b).

### External-only protocol (this notebook)

For **every** held-out truth connectivity `k` of every panel:

`candidate_sources_without_train(k) = candidate_sources(k) − {TRAIN}`

| label | meaning |
|---|---|
| `TRAIN_REACHABLE` | k has TRAIN provenance: **diagnostic only, never counted** |
| `COCONUT_REACHABLE` | k has COCONUT provenance (finalized universe, current filter policy) |
| `PUBCHEM_REACHABLE` | k is in the local PubChem key-membership index: **PUBCHEM_KEY_MEMBERSHIP_ESTIMATE** |
| `EXTERNAL_REACHABLE` | any external source remains after removing TRAIN |
| `EXTERNAL_UNREACHABLE` | none remains: TRAIN-only truth |

**Database coverage failure ≠ mass retrieval failure.** Not externally reachable means fix the DATABASE. Reachable but outside the ppm window means fix precursor, adduct or mass handling. The old diagnostics of `13_colab_candidate_recall.ipynb` remain **LEGACY_DIAGNOSTIC_ONLY**.

## Decision rules: declared BEFORE any result is computed

Thresholds come from `coverage.decision`. The next code cell writes them (with the policy identity) to `decision_rules.json` *before* any coverage number exists. **Do not tune thresholds after seeing results.** The rules are evaluated on the **decision panel**: HOST (enveda-np-examples) when available, else NP_PANEL (structure-only NP-likeness). TIMSTOF_DEV (formerly `host_like`) is an instrument subset and never governs a decision.

| rule | condition (decision panel) | decision |
|---|---|---|
| **A** | PubChem adds ≥ `rule_a_min_incremental_pp` (10) percentage points of external reachability over COCONUT | `BUILD_PUBCHEM_TIER1_NEXT` |
| **B** | reachability ≥ `rule_b_min_reachability` AND the `rule_b_statistic` of `rule_b_pool` at ±`rule_b_ppm` ppm ≥ `rule_b_large_pool_threshold` | `BUILD_FORMULA_PRUNING_BEFORE_RANKING` |
| **C** | charge / salt rescue recovers ≥ `rule_c_min_rescued_fraction` of the truths reachable before filters | `REBUILD_COCONUT_WITH_RESCUE_POLICY` |
| **D** | COCONUT reachability ≥ `rule_d_min_coconut_reachability` AND filter-lost fraction ≤ `rule_d_max_lost_fraction` | `PROCEED_TO_ANALOG_BASELINE` |

* Only Rule A's 10 pp is given by the project. The rule B / C / D thresholds ("A8 thresholds") have **no** project value (the repository was searched; "A8" there is the molecule aggregator). While any is `null` every rule is still evaluated and shown, but the decision is **`BLOCKED_A8_THRESHOLDS`**: no final decision is made.
* Several rules may fire; the first fired rule in `priority` (A, C, B, D) is the mechanical decision.
* Without a decision panel (HOST missing and NP_PANEL not declared) the decision is `DECISION_PANEL_UNAVAILABLE`.

## Protocol validity (v2)

| case | universe | validity |
|---|---|---|
| **A** | no external source | `INVALID` |
| **B** | legacy, provenance verified in this run, **not adopted** | `VALID_FOR_READ_ONLY_COVERAGE` + warning (regimes cannot bind to it, so this notebook stops at the regime check) |
| **C** | modern manifest v3, or legacy **adopted** by notebook 11a | `VALID` |
| **D** | contradiction (files changed since adoption, failed verification, regimes bound to another universe) | `INVALID` |

Regimes (notebook 11b): **C1** = has a mirror-aware reference; else **C2** = externally reachable; else **C3**; **UNRESOLVED** = no mirror-aware evidence. Reported per panel with natural counts.

In [ ]:
# bootstrap: Drive, repository (cloned by notebook 10), config, seeds -- nothing is computed yet
import os, sys, json
from pathlib import Path
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
DRIVE_ROOT = Path(os.environ.setdefault('ENVEDA_DRIVE_ROOT', '/content/drive/MyDrive/EnvedaCASMI'))
_repo_candidates = [os.environ.get('ENVEDA_REPO_ROOT'), str(Path('/content', 'Enveda')), str(DRIVE_ROOT / 'repo')]
REPO_ROOT = next((Path(p) for p in _repo_candidates if p and (Path(p) / 'src' / 'casmi').is_dir()), None)
if REPO_ROOT is None:
    raise FileNotFoundError(f'CASMI repository not found in {_repo_candidates}: run notebook 10 (it clones the repository) or set ENVEDA_REPO_ROOT')
os.environ['ENVEDA_REPO_ROOT'] = str(REPO_ROOT)
if str(REPO_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / 'src'))
from casmi.workspace.environment import ensure_packages, set_seeds, system_report
ensure_packages({'rdkit': 'rdkit', 'yaml': 'pyyaml'})
from casmi.workspace.config import load_v2_config, input_path
CFG, P = load_v2_config(REPO_ROOT / 'configs' / 'casmi_v2_colab.yaml')
P.ensure()
SEED = set_seeds(CFG['runtime']['seed'])
NOTEBOOK = '13_candidate_coverage'
COV, V2 = CFG['coverage'], CFG['validation_v2']
REPORT = P.drive_root / COV['report_dir']
REPORT.mkdir(parents=True, exist_ok=True)
UNIVERSE_ROOT = P.candidate_db_dir / 'universe'
WORK = P.candidate_db_dir / 'universe_work'
print('repo', REPO_ROOT, '| scratch', P.scratch_dir, '| reports', REPORT)
print(system_report())

In [ ]:
# decision policy + A8 threshold status + decision-panel order fixed and SAVED before any coverage number exists
from datetime import datetime, timezone
from casmi.validation.external_coverage import COVERAGE_GATE_VERSION, DecisionPolicy, write_json
from casmi.validation.protocol_v2 import BLOCKED_A8_THRESHOLDS, PROTOCOL_V2_VERSION, a8_threshold_status, policy_identity_v2
POLICY = DecisionPolicy.from_dict(COV['decision'])
PANEL_ORDER = list((CFG.get('panels') or {}).get('decision_panel_order') or ['HOST', 'NP_PANEL'])
A8 = a8_threshold_status(POLICY)
POLICY_ID = policy_identity_v2(POLICY, A8, PANEL_ORDER)
DECISION_RULES = {
    'coverage_gate_version': COVERAGE_GATE_VERSION, 'protocol_version': PROTOCOL_V2_VERSION, 'declared_at': datetime.now(timezone.utc).isoformat(),
    'policy': dict(POLICY.__dict__), 'policy_identity': POLICY_ID, 'a8_thresholds': A8, 'decision_panel_order': PANEL_ORDER,
    'rules': {'A': 'PubChem adds >= rule_a_min_incremental_pp pp of external reachability over COCONUT on the decision panel -> BUILD_PUBCHEM_TIER1_NEXT',
              'B': 'reachability >= rule_b_min_reachability AND pool statistic at rule_b_ppm >= rule_b_large_pool_threshold -> BUILD_FORMULA_PRUNING_BEFORE_RANKING',
              'C': 'rescue recovers >= rule_c_min_rescued_fraction of truths reachable before filters -> REBUILD_COCONUT_WITH_RESCUE_POLICY',
              'D': 'COCONUT reachability >= rule_d_min_coconut_reachability AND lost fraction <= rule_d_max_lost_fraction -> PROCEED_TO_ANALOG_BASELINE'},
    'unset_threshold_behaviour': 'rule NOT_EVALUATED; any null A8 threshold -> BLOCKED_A8_THRESHOLDS (no final decision)'}
write_json(REPORT / 'decision_rules.json', DECISION_RULES)
print(json.dumps(DECISION_RULES, indent=2, default=str))
if A8['status'] == BLOCKED_A8_THRESHOLDS:
    print('*** BLOCKED_A8_THRESHOLDS: rules B/C/D lack thresholds', A8['missing'], '-- results are shown, no final decision is made')

In [ ]:
# universe identity (modern / adopted / legacy / contradiction) + protocol validity cases A-D (read-only)
from casmi.candidates.legacy_adoption import effective_universe_identity, verify_legacy_universe
from casmi.candidates.universe import universe_manifest_status
from casmi.validation.protocol_v2 import INVALID, protocol_validity_v2
if not (UNIVERSE_ROOT / 'universe_manifest.json').exists():
    raise FileNotFoundError(f'no finalized universe under {UNIVERSE_ROOT}: run notebook 12 (Stage A / B / C) once')
UID = effective_universe_identity(UNIVERSE_ROOT)
MSTATUS = universe_manifest_status(UNIVERSE_ROOT)
VERIFY = verify_legacy_universe(UNIVERSE_ROOT) if UID['kind'] == 'legacy_unadopted' else None   # case B (read-only verification)
PROTO = protocol_validity_v2(UID, VERIFY)
EXT_SOURCES = PROTO['external_sources']
print(json.dumps({'universe': {k: UID.get(k) for k in ('kind', 'build_id', 'universe_status', 'n_candidates', 'contents_verified',
                                                       'reproducible_build_id_available', 'problems')},
                  'legacy_manifest_detected': MSTATUS.get('legacy_manifest_detected'), **PROTO}, indent=2, default=str))
if PROTO['protocol_validity'] == INVALID:
    print('*** protocol INVALID (case', PROTO['protocol_case'], '):', PROTO['protocol_reasons'])

In [ ]:
# validation panels (ALL_DEV / TIMSTOF_DEV / HOST / NP_PANEL; never merged); decision panel HOST, else NP_PANEL
import numpy as np, pandas as pd
from casmi.validation.panels import panels_for_run
DEV_COLS = ['query_id', 'true_connectivity_key', 'fold', 'neutral_mass', 'ionization_mode', 'adduct', 'instrument_type', 'exact_mass_true']
dev_path = input_path(CFG, P, 'dev_queries')
if not dev_path.exists():
    raise FileNotFoundError(f'grouped validation queries missing: {dev_path} (upload data/interim/candidate_generation/dev_queries.parquet)')
dev = pd.read_parquet(dev_path)
missing = {'query_id', 'true_connectivity_key', 'fold', 'neutral_mass'} - set(dev.columns)
if missing:
    raise KeyError(f'dev_queries lacks required columns {sorted(missing)}')
dev = dev[[c for c in DEV_COLS if c in dev.columns]]
dev['query_id'] = dev['query_id'].astype(str)
PAN = panels_for_run(CFG, P.drive_root, dev)
Q, PANEL_STATUS, DECISION_PANEL = PAN['queries'], PAN['status'], PAN['decision']
display(pd.DataFrame(PANEL_STATUS).T[['status', 'role', 'reason', 'n_queries', 'n_molecules']])
print('decision panel:', DECISION_PANEL['decision_panel'], '|', DECISION_PANEL['reason'])
write_json(REPORT / 'panel_status.json', {'panels': PANEL_STATUS, 'decision_panel': DECISION_PANEL,
                                          'panel_manifest_identity': PAN['manifest']['panel_manifest_identity']})

In [ ]:
# REQUIRE the v2 regime manifest of notebook 11b, bound to THIS identity; stop early otherwise (REGIME_REBUILD_REQUIRED)
from casmi.candidates.pubchem_membership import PubChemMembershipIndex
from casmi.validation.regimes_v2 import (REGIME_CURRENT, RegimeRebuildRequired, check_regime_manifest, load_reference_evidence,
                                         regime_identity_from_run)
FOLDS_PATH = input_path(CFG, P, 'connectivity_folds')
folds = pd.read_parquet(FOLDS_PATH)
ME = V2['mirror_aware_evidence']
_, MIRROR_INFO = load_reference_evidence([P.drive_root / f for f in ME['files']], ME['flag_column'], 'mirror_aware')
PC_DIR = P.drive_root / COV['pubchem_membership']['index_dir']
PCI = PubChemMembershipIndex(PC_DIR) if PubChemMembershipIndex.available(PC_DIR) else None
EXPECTED_ID = regime_identity_from_run(cfg=CFG, repo_root=REPO_ROOT, dev=dev, dev_path=dev_path, folds=folds, folds_path=FOLDS_PATH,
                                       universe_identity=UID, mirror_info=MIRROR_INFO, pubchem_index=PCI, host_path=PAN['host_path'],
                                       panel_manifest_identity=PAN['manifest']['panel_manifest_identity'])
REGCHK = check_regime_manifest(P.drive_root / V2['manifest_path'], EXPECTED_ID, V2['strict_git_commit'])
print('regime manifest:', REGCHK['status'], '| mismatches:', [m['field'] for m in REGCHK['mismatches']], '| warnings:', REGCHK['warnings'])
if REGCHK['status'] != REGIME_CURRENT:
    write_json(REPORT / 'coverage_gate_summary.json', {'notebook': NOTEBOOK, 'completed': False, 'decision': 'REGIME_REBUILD_REQUIRED',
                                                       'protocol_valid': False, 'regime_check': {k: REGCHK[k] for k in ('status', 'mismatches', 'warnings')},
                                                       'finished_at': datetime.now(timezone.utc).isoformat()})
    raise RegimeRebuildRequired(f"REGIME_REBUILD_REQUIRED ({REGCHK['status']}): {[m['field'] for m in REGCHK['mismatches']]} -- run 11a/11b")
REGIME_SIGNATURE = REGCHK['manifest']['regime_signature']
RECORDS = pd.read_parquet(P.drive_root / V2['records_path'])
RMAP = RECORDS.drop_duplicates('connectivity_key').set_index('connectivity_key')['regime']
Q['regime'] = Q['connectivity_key'].map(RMAP).fillna('NOT_IN_REGIME_TABLE')
N_NOT_IN = int((Q['regime'] == 'NOT_IN_REGIME_TABLE').sum())
assert N_NOT_IN == 0, f'{N_NOT_IN} panel queries have no regime although the manifest is current -- re-run 11b'
print('regime signature', REGIME_SIGNATURE)

In [ ]:
# membership + leakage-free reachability (TRAIN provenance removed for every held-out truth)
from casmi.candidates.pubchem_membership import membership_summary
from casmi.validation.external_coverage import (build_external_membership, c2_external_protocol, classify_external_reachability,
                                                 molecule_table, stratify_by_mass_bin, universe_truth_rows)
MOL = molecule_table(Q)
TRUTH_ROWS = universe_truth_rows(UNIVERSE_ROOT, MOL['connectivity_key'])
print('PubChem membership:', 'AVAILABLE (PUBCHEM_KEY_MEMBERSHIP_ESTIMATE)' if PCI else 'NOT_AVAILABLE (PUBCHEM_MEMBERSHIP_SOURCE_MISSING until built; Rule A NOT_EVALUATED)')
MEMB = classify_external_reachability(build_external_membership(MOL['connectivity_key'], TRUTH_ROWS, PCI))
MOL = c2_external_protocol(MOL.merge(MEMB, on='connectivity_key', how='left', validate='many_to_one'))
MOL['mass_bin'] = stratify_by_mass_bin(MOL['truth_exact_mass'].fillna(MOL['median_query_neutral_mass']), CFG['validation']['mass_bins_da'])
MOL['truth_formula'] = MOL['connectivity_key'].map(TRUTH_ROWS.set_index('connectivity_key')['molecular_formula'])
print('C2 truths without any external source (must be 0):', int(MOL['legacy_c2_train_only'].sum()))
diag = MOL.groupby('panel')[['in_train', 'membership_in_full_train_union', 'external_reachable']].mean()
display(diag.rename(columns={'in_train': 'train_membership_DIAGNOSTIC', 'membership_in_full_train_union': 'in_full_universe_DIAGNOSTIC_NOT_COVERAGE'}))

In [ ]:
# external reachability report: panel / regime (C1 / C2 / C3 / UNRESOLVED, natural counts) / mass bin (TRAIN never counts)
from casmi.validation.external_coverage import reachability_summary
from casmi.validation.regimes_v2 import regime_counts_by_panel
REGIME_COUNTS = regime_counts_by_panel(RECORDS)
display(REGIME_COUNTS.pivot(index='panel', columns='regime', values='n_molecules').fillna(0).astype(int))
REACH_PANEL = reachability_summary(MOL, by=('panel',))
REACH_REGIME = reachability_summary(MOL, by=('panel', 'regime'))
REACH_MASS = reachability_summary(MOL, by=('panel', 'mass_bin'))
display(REACH_PANEL)
display(REACH_REGIME)
MOL_OUT = MOL.assign(candidate_sources=MOL['candidate_sources'].map(';'.join),
                     candidate_sources_without_train=MOL['candidate_sources_without_train'].map(';'.join))
MOL_OUT.to_parquet(REPORT / 'external_reachability_by_molecule.parquet', index=False)
pd.concat([REACH_PANEL.assign(grouping='panel'), REACH_REGIME.assign(grouping='panel+regime')], ignore_index=True).to_csv(
    REPORT / 'external_reachability_summary.csv', index=False)
REACH_MASS.to_csv(REPORT / 'reachability_by_mass_bin.csv', index=False)
REGIME_COUNTS.to_csv(REPORT / 'regime_counts_by_panel.csv', index=False)

In [ ]:
# optional PubChem key membership (local index only; NOT_AVAILABLE without it)
PC_SUMMARY = membership_summary(PCI, MOL['connectivity_key'].unique() if PCI else None)
INCR = REACH_MASS[['panel', 'mass_bin', 'n_molecules', 'coconut_reachability', 'pubchem_reachability', 'union_external_reachability',
                   'incremental_pubchem_over_coconut_pp', 'pubchem_label']]
write_json(REPORT / 'pubchem_membership_summary.json', PC_SUMMARY)
INCR.to_csv(REPORT / 'pubchem_incremental_coverage.csv', index=False)
print(json.dumps(PC_SUMMARY, indent=2, default=str))
display(INCR)

In [ ]:
# mass-window coverage at 2 / 5 / 10 ppm: database coverage vs mass retrieval (no ranking)
from casmi.candidates.mass_index import CandidateMassIndex
from casmi.candidates.pubchem_tier1 import Tier1MassIndex
from casmi.validation.external_coverage import evaluate_mass_window_coverage, external_candidate_mask, window_outcome_summary
IDX = CandidateMassIndex.load(UNIVERSE_ROOT / 'index')
MASKS = external_candidate_mask(UNIVERSE_ROOT)
print({k: int(v.sum()) for k, v in MASKS.items()}, '| candidates', IDX.n_candidates)
TIER1_TABLE = P.drive_root / COV['pubchem_tier1']['table']
TIER1 = Tier1MassIndex.from_table(TIER1_TABLE) if TIER1_TABLE.exists() else None
print('projected PubChem Tier-1 pools:', 'AVAILABLE (RAW CID counts, not connectivities)' if TIER1 else 'NOT_AVAILABLE')
cid_of = TRUTH_ROWS.set_index('connectivity_key')['candidate_id']
QW = Q.merge(MOL[['panel', 'connectivity_key', 'external_reachable', 'mass_bin']], on=['panel', 'connectivity_key'], how='left')
QW['truth_candidate_id'] = QW['connectivity_key'].map(cid_of).fillna(-1).astype(np.int64)
WIN = evaluate_mass_window_coverage(IDX, QW, COV['ppm_values'], external_mask=MASKS['external'], tier1_index=TIER1)
WIN = WIN.merge(QW[['panel', 'query_id', 'mass_bin', 'regime']], on=['panel', 'query_id'], how='left')
WIN_SUMMARY = window_outcome_summary(WIN)
display(WIN_SUMMARY)

In [ ]:
# candidate-pool density: finalized connectivity counts vs projected RAW PubChem Tier-1 counts (never added together)
from casmi.validation.external_coverage import summarize_pool_density
POOL = summarize_pool_density(WIN, by=('panel', 'ppm'))
display(POOL[POOL['n'] > 0])
WIN.to_parquet(REPORT / 'mass_pool_density.parquet', index=False)
POOL.to_csv(REPORT / 'mass_pool_density_summary.csv', index=False)
WIN_SUMMARY.to_csv(REPORT / 'mass_window_outcomes.csv', index=False)
FAIL_KIND = WIN.groupby(['panel', 'ppm'])['coverage_outcome'].value_counts(normalize=True).unstack(fill_value=0)
print('database_coverage_failure = NOT externally reachable  -> fix the DATABASE')
print('mass_retrieval_failure    = reachable, outside the ppm window -> fix precursor / adduct / MASS handling')
display(FAIL_KIND.round(4))

In [ ]:
# filter-loss audit inputs: external records the CURRENT policy dropped -- modern ACTIVE builds AND legacy chunk markers
from casmi.candidates.legacy_adoption import discover_stage_a_builds, select_stage_a_namespaces
from casmi.candidates.stage_a import StructureEngine
from casmi.validation.external_coverage import collect_lost_records, key_unkeyed_lost_records
from casmi.workspace.parallel import resolve_n_jobs
STAGE_A = discover_stage_a_builds(WORK)
NS, NS_SELECTION = select_stage_a_namespaces(STAGE_A, UID, sources=EXT_SOURCES)
display(pd.DataFrame(NS_SELECTION))
if not NS:
    print('*** no Stage-A build matches the finalized universe: filter-loss / rescue metrics stay unavailable (Rules C / D NOT_EVALUATED)')
LOST = collect_lost_records(NS, sources=EXT_SOURCES)
display(LOST.groupby(['source', 'loss_stage', 'loss_categories']).size().rename('n').reset_index())
N_JOBS = resolve_n_jobs(COV['rescue']['n_jobs'])
if COV['rescue']['canonicalize_prefiltered'] and len(LOST):
    with StructureEngine('universe_minimal', N_JOBS) as ENG:      # training canonicalizer, current policy (no cleanup)
        LOST = key_unkeyed_lost_records(LOST, ENG)
n_keyed = int(LOST['connectivity_key'].notna().sum()) if len(LOST) else 0
print(f'lost records: {len(LOST):,} | with a connectivity key: {n_keyed:,} | workers {N_JOBS}')

In [ ]:
# current_policy vs rescue_policy per panel (lost / rescued fraction, newly reachable, rescue gain pp); universe untouched
from casmi.candidates.universe import UniverseBuildConfig
from casmi.validation.external_coverage import audit_filter_losses, filter_loss_summary, filter_policy_comparison, rescue_lost_records
UCFG = UniverseBuildConfig.from_dict(CFG['universe'])
with StructureEngine('universe_minimal', N_JOBS) as ENG:
    RESCUED = rescue_lost_records(LOST, UCFG.filters, ENG, policy='rescue_policy')
if len(RESCUED):
    display(RESCUED.groupby(['rescue_kind', 'passes_after_rescue']).size().rename('n').reset_index())
LOSS = audit_filter_losses(MOL, LOST, RESCUED, external_sources=EXT_SOURCES, mass_bins=CFG['validation']['mass_bins_da'])
LOSS_SUMMARY = filter_loss_summary(LOSS)
POLICY_CMP = filter_policy_comparison(LOSS)
display(LOSS_SUMMARY)
display(POLICY_CMP)
if len(LOSS):
    LOSS.to_parquet(REPORT / 'filter_loss_by_molecule.parquet', index=False)
LOSS_SUMMARY.merge(POLICY_CMP, on=['panel', 'external_source'], how='left').to_csv(REPORT / 'filter_loss_summary.csv', index=False)
RESCUED.to_parquet(REPORT / 'rescue_attempts.parquet', index=False)

In [ ]:
# mass-filter audit: truths outside the CURRENT external mass range (never changed automatically)
from casmi.validation.external_coverage import inference_mass_range, mass_range_audit, read_optional_table
MASS_RANGE = mass_range_audit(MOL, UCFG.filters)
display(MASS_RANGE)
INF_PATH = COV.get('inference_metadata_path')
INF_RANGE = inference_mass_range(read_optional_table(P.drive_root / INF_PATH) if INF_PATH else None)
print('inference-safe neutral-mass range:', json.dumps(INF_RANGE, default=str))
MASS_RANGE.to_csv(REPORT / 'mass_range_audit.csv', index=False)
write_json(REPORT / 'inference_mass_range.json', INF_RANGE)

In [ ]:
# adduct / charge audit: only adducts PRESENT in the data; [M]+ style adducts surfaced, never assumed
from casmi.validation.external_coverage import adduct_charge_audit
ADDUCTS = adduct_charge_audit(Q.loc[Q['panel'] == 'ALL_DEV', 'adduct']) if Q['adduct'].notna().any() else pd.DataFrame()
display(ADDUCTS.head(40))
PERM = ADDUCTS[ADDUCTS['permanently_charged_compatible']] if len(ADDUCTS) else ADDUCTS
print('adducts compatible with a PERMANENTLY charged candidate:', PERM['adduct'].tolist() if len(PERM) else 'none present in the data')
ADDUCTS.to_csv(REPORT / 'adduct_charge_audit.csv', index=False)

In [ ]:
# representative SMILES -> EXACT project canonicalizer -> connectivity key == stored key ? (resumable per bucket)
from casmi.validation.smiles_roundtrip import run_universe_roundtrip
RT = COV['roundtrip']
RT_FAILS, RT_SUMMARY = run_universe_roundtrip(UNIVERSE_ROOT, REPORT / 'roundtrip_parts', n_jobs=resolve_n_jobs(RT['n_jobs']),
                                              max_per_bucket=RT['max_per_bucket'], seed=CFG['validation']['split_seed'])
RT_FAILS.to_parquet(REPORT / 'smiles_roundtrip_failures.parquet', index=False)
write_json(REPORT / 'smiles_roundtrip_summary.json', RT_SUMMARY)
print(json.dumps({k: RT_SUMMARY[k] for k in ('n_candidates', 'n_roundtrip_ok', 'n_roundtrip_fail', 'failure_rate')}, indent=2))
display(pd.DataFrame(RT_SUMMARY['by_stratum']).T)

In [ ]:
# actionable diagnosis of every round-trip failure + deployment check (no representative replaced, nothing blacklisted)
from casmi.validation.protocol_v2 import roundtrip_deployment_check
from casmi.validation.roundtrip_diagnostics import collect_source_records, diagnose_roundtrip_failures, diagnosis_summary
SRC_RECS = collect_source_records(NS, RT_FAILS['stored_connectivity_key'], sources=EXT_SOURCES)
RT_DIAG = diagnose_roundtrip_failures(RT_FAILS, SRC_RECS)
RT_DIAG_SUMMARY = diagnosis_summary(RT_DIAG)
RT_DEPLOY = roundtrip_deployment_check(RT_FAILS, exhaustive=RT['max_per_bucket'] is None)
RT_DIAG.to_parquet(REPORT / 'smiles_roundtrip_diagnosis.parquet', index=False)
write_json(REPORT / 'smiles_roundtrip_diagnosis.json', {**RT_DIAG_SUMMARY, 'deployment': RT_DEPLOY})
print(json.dumps(RT_DIAG_SUMMARY, indent=2))
print('deployment:', RT_DEPLOY['status'], '|', RT_DEPLOY['reason'])
display(RT_DIAG[['candidate_id', 'stored_connectivity_key', 'outcome', 'n_source_records', 'evidence', 'proposed_smiles']])

In [ ]:
# official-metric parity: only with a COMPETITION-PROVIDED canonicalizer file (none in the repository)
from casmi.validation.external_coverage import official_metric_parity
from casmi.validation.smiles_roundtrip import project_keys
OM = COV.get('official_metric') or {}
OM_PATH = (P.drive_root / OM['module_path']) if OM.get('module_path') else None
SAMPLE = (TRUTH_ROWS.sort_values('connectivity_key')['representative_smiles'].head(int(OM.get('sample_size') or 500)).tolist()
          + RT_FAILS['representative_smiles'].head(100).tolist())          # deterministic sample + round-trip failures
OFFICIAL = official_metric_parity(OM_PATH, OM.get('canonicalizer_function'), SAMPLE, project_keyfn=lambda s: project_keys([s])[0][0])
write_json(REPORT / 'official_metric_parity.json', OFFICIAL)
print('OFFICIAL_METRIC_PARITY =', OFFICIAL['OFFICIAL_METRIC_PARITY'], '|', OFFICIAL.get('reason', ''))

In [ ]:
# public-notebook baseline: coverage gap vs ranking gap through the documented INTERFACE (LOCAL artifact only; none exists)
from casmi.validation.external_coverage import public_baseline_coverage
from casmi.validation.public_baseline import PublicBaselineSpec, load_public_baseline
PB_SPEC = PublicBaselineSpec.from_dict(COV.get('public_baseline'))
PB_INTERFACE, PUB = load_public_baseline(PB_SPEC, P.drive_root, keyfn=lambda s: project_keys([s])[0][0])
PUB_STATUS, PUB_CMP = public_baseline_coverage(PUB, Q if PUB is not None else None, WIN, ppm=PB_SPEC.ppm)
PUB_STATUS = {**PUB_STATUS, 'interface': PB_INTERFACE}
write_json(REPORT / 'public_baseline_coverage.json', PUB_STATUS)
PUB_CMP.to_csv(REPORT / 'public_baseline_coverage_comparison.csv', index=False)
print('PUBLIC_BASELINE_COVERAGE =', PUB_STATUS['PUBLIC_BASELINE_COVERAGE'], '| interface:', PB_INTERFACE['status'], PB_INTERFACE.get('reason', ''))

In [ ]:
# ESTIMATE_ONLY planning helper: linear extrapolation of an OBSERVED Stage-A throughput (modern markers carry timings)
from casmi.validation.external_coverage import ESTIMATE_ONLY, estimate_runtime, observed_stage_a_throughput
TP = {s: observed_stage_a_throughput(r['namespace'], s) for s, recs in STAGE_A.items() for r in recs
      if not r['legacy'] and r['status'] == 'complete'}
print(json.dumps(TP, indent=2))
rps = (TP.get('COCONUT') or {}).get('records_per_second')
EST = ({f'{n:,}_records': estimate_runtime(n, rps) for n in (1_000_000, 10_000_000, 120_000_000)} if rps
       else {'label': ESTIMATE_ONLY, 'status': 'no observed throughput (legacy chunk markers carry no timings)'})
print(ESTIMATE_ONLY, '(same worker count as the observed run):')
print(json.dumps(EST, indent=2, default=str))

In [ ]:
# mechanical evaluation of the PRE-DECLARED rules on the decision panel + v2 overlay (protocol, A8, panel, round trip)
from casmi.validation.external_coverage import decision_metrics, evaluate_decision_rules
from casmi.validation.protocol_v2 import final_decision_v2
PROTO = protocol_validity_v2(UID, VERIFY, regime_universe_build_id=REGCHK['manifest']['identity'].get('universe_build_id'), n_population_rows=len(Q))
POLICY_RUN = DecisionPolicy.from_dict({**COV['decision'], 'natural_product_panel': DECISION_PANEL['decision_panel']})
METRICS = decision_metrics(REACH_PANEL, POOL, LOSS_SUMMARY, POLICY_RUN)
RULES = evaluate_decision_rules(METRICS, POLICY_RUN, protocol_valid=PROTO['protocol_valid_for_coverage'])
FINAL = final_decision_v2(RULES, PROTO, A8, DECISION_PANEL, RT_DEPLOY)
write_json(REPORT / 'decision_rules.json', {**DECISION_RULES, 'evaluation': RULES, 'final': FINAL, 'metrics': METRICS})
GATE = {'notebook': NOTEBOOK, 'coverage_gate_version': COVERAGE_GATE_VERSION, 'protocol_version': PROTOCOL_V2_VERSION, 'completed': True,
        **PROTO, **FINAL, 'fired_rules': RULES['fired_rules'], 'rules': RULES['rules'], 'metrics': METRICS, 'a8_thresholds': A8,
        'policy_identity': POLICY_ID, 'regime_signature': REGIME_SIGNATURE, 'regime_check_warnings': REGCHK['warnings'],
        'panel_manifest_identity': PAN['manifest']['panel_manifest_identity'], 'panel_status': PANEL_STATUS,
        'universe': {k: UID.get(k) for k in ('kind', 'build_id', 'universe_status', 'n_candidates', 'contents_verified')},
        'stage_a_selection': NS_SELECTION, 'pubchem_membership': PC_SUMMARY.get('status'),
        'official_metric_parity': OFFICIAL['OFFICIAL_METRIC_PARITY'], 'public_baseline_coverage': PUB_STATUS['PUBLIC_BASELINE_COVERAGE'],
        'smiles_roundtrip_failure_rate': RT_SUMMARY['failure_rate'], 'roundtrip_diagnosis': RT_DIAG_SUMMARY, 'roundtrip_deployment': RT_DEPLOY,
        'reachability_by_panel': REACH_PANEL.to_dict(orient='records'), 'regime_counts_by_panel': REGIME_COUNTS.to_dict(orient='records'),
        'ranking_metrics': 'NOT COMPUTED (coverage gate only; ranking starts in notebook 14)', 'finished_at': datetime.now(timezone.utc).isoformat()}
write_json(REPORT / 'coverage_gate_summary.json', GATE)
print('DECISION:', GATE['decision'], '| mechanical:', GATE['mechanical_decision'], '| fired:', GATE['fired_rules'])
print('protocol:', GATE['protocol_validity'], '(case', GATE['protocol_case'], ') | decision panel:', GATE['decision_panel'],
      '| permits ranking:', GATE['permits_ranking'], GATE['ranking_blockers'])
for r, v in RULES['rules'].items():
    print('  Rule', r, '|', v['status'], '|', v['reason'])

In [ ]:
# figures 1-2: external reachability by panel (COCONUT / PubChem / union); incremental PubChem over COCONUT by mass bin
import matplotlib.pyplot as plt
from casmi.validation.external_coverage import mass_bin_labels
FIG = REPORT / 'figures'
FIG.mkdir(parents=True, exist_ok=True)
ORDER = mass_bin_labels(CFG['validation']['mass_bins_da']) + ['unknown']
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
REACH_PANEL.set_index('panel')[['coconut_reachability', 'pubchem_reachability', 'union_external_reachability']].plot.bar(
    ax=ax[0], title='external reachability by panel (TRAIN never counts)')
ax[0].set_ylim(0, 1)
for p, g in REACH_MASS.groupby('panel'):
    g = g.set_index('mass_bin').reindex([o for o in ORDER if o in set(g['mass_bin'])])
    ax[1].plot(g.index, g['incremental_pubchem_over_coconut_pp'], marker='o', label=p)
ax[1].set_title('incremental PubChem reachability over COCONUT (pp) by mass bin')
ax[1].tick_params(axis='x', rotation=45)
ax[1].legend()
fig.tight_layout()
fig.savefig(FIG / 'external_reachability.png', dpi=120)

In [ ]:
# figures 3-4: candidate-pool sizes at 2 / 5 / 10 ppm; current finalized vs projected PubChem Tier-1 (RAW) density
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for ppm, g in WIN[WIN['panel'] == 'ALL_DEV'].groupby('ppm'):
    ax[0].hist(np.log10(g['full_current_universe_pool_size'].clip(lower=1)), bins=60, histtype='step', label=f'{ppm} ppm')
ax[0].set_xlabel('log10 pool size (TRAIN u COCONUT, finalized connectivities)')
ax[0].set_title('candidate-pool sizes (ALL_DEV)')
ax[0].legend()
d = POOL[(POOL['n'] > 0) & POOL['pool'].isin(['full_current_universe_pool_size', 'projected_pubchem_tier1_pool_size'])]
if len(d):
    d.pivot_table(index=['panel', 'ppm'], columns='pool', values='median').plot.bar(
        ax=ax[1], logy=True, title='median pool: finalized connectivities vs projected PubChem Tier-1 RAW CIDs')
fig.tight_layout()
fig.savefig(FIG / 'pool_density.png', dpi=120)

In [ ]:
# figures 5-6: filter-loss / rescue waterfall (decision panel, else ALL_DEV) and representative-SMILES round trip by stratum
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
FIG_PANEL = DECISION_PANEL['decision_panel'] or 'ALL_DEV'
ls = LOSS_SUMMARY[LOSS_SUMMARY['panel'] == FIG_PANEL] if len(LOSS_SUMMARY) else LOSS_SUMMARY
if len(ls):
    s = ls[['reachable_before_filters', 'lost_count', 'rescued_count', 'newly_reachable_by_rescue', 'reachable_after_rescue']].sum()
    steps = {'reachable before filters': s['reachable_before_filters'], 'lost (current policy)': -s['lost_count'],
             'rescued (rescue policy)': s['rescued_count'], 'newly reachable (rescue)': s['newly_reachable_by_rescue'],
             'reachable after rescue': s['reachable_after_rescue']}
    ax[0].bar(list(steps), list(steps.values()), color=['grey', 'tab:red', 'tab:green', 'tab:blue', 'black'])
    ax[0].set_title(f'filter loss / rescue ({FIG_PANEL} truths)')
    ax[0].tick_params(axis='x', rotation=20)
rt = pd.DataFrame(RT_SUMMARY['by_stratum']).T
if len(rt):
    rt[['n_ok', 'n_fail']].astype(float).plot.bar(stacked=True, ax=ax[1], logy=True, title='representative-SMILES round trip by stratum')
fig.tight_layout()
fig.savefig(FIG / 'filter_loss_and_roundtrip.png', dpi=120)

In [ ]:
# artifact metadata + BOUND experiment row (git commit, universe, regime signature, panels, policy); no ranking metric logged
from casmi.workspace.artifacts import git_commit, write_metadata
from casmi.workspace.experiments import ExperimentBindingError, log_bound_experiment
write_metadata(REPORT, NOTEBOOK, config={'coverage': COV, 'universe_filters': CFG['universe']['filters'], 'validation_v2': V2}, seed=SEED,
               repo_root=REPO_ROOT, input_paths={'universe': UNIVERSE_ROOT, 'universe_work': WORK, 'dev_queries': dev_path,
                                                 'regime_manifest': P.drive_root / V2['manifest_path'], 'pubchem_membership': PC_DIR},
               schema_version=COVERAGE_GATE_VERSION, extra={'decision': GATE['decision'], 'universe_build_id': UID['build_id'],
                                                            'policy_identity': POLICY_ID, 'regime_signature': REGIME_SIGNATURE})
np_row = REACH_PANEL.set_index('panel')
note = f"{GATE['decision']} | coconut reachability (ALL_DEV) = {np_row['coconut_reachability'].get('ALL_DEV')}"
try:
    log_bound_experiment(P.experiments_dir, 'gateA_external_coverage_v2', notebook=NOTEBOOK, git_commit=git_commit(REPO_ROOT),
                         universe_build_id=UID['build_id'], regime_signature=REGIME_SIGNATURE,
                         panel_manifest_identity=PAN['manifest']['panel_manifest_identity'], policy_identity=POLICY_ID,
                         model_name='none (coverage gate)', status=GATE['decision'], kind='coverage', channels=['coverage'],
                         description='external candidate coverage gate v2 (notebook 13; coverage only)', config_path=CFG['_config_path'], notes=note)
except ExperimentBindingError as e:
    print('experiment row NOT written (unbound):', e)
print('saved under', REPORT)
for f in sorted(REPORT.glob('*')):
    print(' ', f.name)

## Final decision summary

The `c-decision` cell prints the decision from `coverage_gate_summary.json`:

| decision | next step |
|---|---|
| `REGIME_REBUILD_REQUIRED` (stopped at `c-regimes`) | run 11a (if the universe is legacy) and 11b, then re-run this notebook |
| `BLOCKED_A8_THRESHOLDS` | declare the rule B / C / D thresholds in `coverage.decision` **before** re-running (not after looking at results) |
| `DECISION_PANEL_UNAVAILABLE` | upload the HOST file (`panels.HOST.path`) or declare an NP_PANEL threshold, re-run 11b, then this notebook |
| `BUILD_PUBCHEM_TIER1_NEXT` | build the PubChem Tier-1 table (`casmi.candidates.pubchem_tier1`), then a deliberate new external universe |
| `REBUILD_COCONUT_WITH_RESCUE_POLICY` | set `universe.cleanup_policy: rescue_policy` and re-run notebook 12; old builds are kept |
| `BUILD_FORMULA_PRUNING_BEFORE_RANKING` | pool density is the bottleneck: formula / pruning work before any ranker |
| `PROCEED_TO_ANALOG_BASELINE` | run `14_colab_analog_baseline.ipynb` only if `permits_ranking` is true (protocol VALID, round-trip deployment OK) |
| `UNDETERMINED_THRESHOLDS_OR_INPUTS_MISSING` / `NO_RULE_FIRED` | inputs missing / no rule fired: read every rule's status |
| `PROTOCOL_INVALID` | no external source, or a universe / regime contradiction: read `protocol_reasons` |

Notebook 14 refuses to run unless the gate completed, the protocol is VALID, the regimes are current and bound to the same universe, the decision permits ranking, the round-trip deployment check is OK and a decision panel exists.